# To-Do Task Manager (Flask Mini Project)


In [1]:
!pip install flask

In [2]:
#create the flask app
from flask import Flask, request

app = Flask(__name__)

tasks = []   # each task is a dictionary: {"id": 1, "title": "...", "done": False}
next_id = 1


@app.route("/")
def home():
    return "Welcome to the To-Do Task Manager!"


@app.route("/tasks")
def show_tasks():
    if len(tasks) == 0:
        return "No tasks yet. Add one using the /add route."
    lines = []
    for t in tasks:
        status = "Done" if t["done"] else "Pending"
        lines.append(f"{t['id']}. {t['title']} - {status}")
    return " , ".join(lines)


@app.route("/add", methods=["POST"])
def add_task():
    global next_id
    title = request.form.get("title", "").strip()

    # Input validation
    if title == "":
        return "Error: Please enter a task title. It cannot be empty.", 400
    if len(title) > 100:
        return "Error: Task title is too long (max 100 characters).", 400

    tasks.append({"id": next_id, "title": title, "done": False})
    next_id += 1
    pending = len([t for t in tasks if not t["done"]])
    return f"Task '{title}' added! You have {pending} pending task(s)."


@app.route("/complete/<int:task_id>", methods=["POST"])
def complete_task(task_id):
    for t in tasks:
        if t["id"] == task_id:
            t["done"] = True
            return f"Task '{t['title']}' marked as done!"
    return "Error: Task not found.", 404


In [3]:
from threading import Thread
from google.colab import output

def run_flask():
  app.run(host='0.0.0.0', port=5000,
          debug=False, use_reloader=False)

Thread(target=run_flask,daemon=True).start()

output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Testing the POST routes

In [4]:
import requests
base = 'http://127.0.0.1:5000'

# valid input
print(requests.post(base + '/add', data={'title': 'Study Flask'}).text)
print(requests.post(base + '/add', data={'title': 'Finish project'}).text)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /add HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /add HTTP/1.1" 200 -


Task 'Study Flask' added! You have 1 pending task(s).
Task 'Finish project' added! You have 2 pending task(s).


In [5]:
# empty input
r = requests.post(base + '/add', data={'title': ''})
print(r.status_code, r.text)

# too long input
r = requests.post(base + '/add', data={'title': 'x' * 101})
print(r.status_code, r.text)

INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /add HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /add HTTP/1.1" 400 -


400 Error: Please enter a task title. It cannot be empty.
400 Error: Task title is too long (max 100 characters).


In [6]:
# mark task 1 as done, then try a task that does not exist
print(requests.post(base + '/complete/1').text)
print(requests.post(base + '/complete/99').text)

INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /complete/1 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "POST /complete/99 HTTP/1.1" 404 -


Task 'Study Flask' marked as done!
Error: Task not found.


In [7]:
# show all tasks
print(requests.get(base + '/tasks').text)

INFO:werkzeug:127.0.0.1 - - [24/Oct/2026 16:11:36] "GET /tasks HTTP/1.1" 200 -


1. Study Flask - Done , 2. Finish project - Pending
